# Finnish Financial Sentiment - Model Training and Evaluation - TurkuNLP/sbert-cased-finnish-paraphrase

## Imports

In [1]:
import datetime
import gc
import psutil
import os

from google.colab import drive

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer
)

from torch import nn
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import StratifiedKFold

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    recall_score,
    precision_score,
    f1_score,
)

from sklearn.model_selection import train_test_split
from datasets import Dataset
import pandas as pd
import numpy as np
import torch

## Mount Google Drive

In [2]:
drive.mount('/content/drive')

Mounted at /content/drive


## GPU/RAM Check

In [3]:
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)

Thu Apr  9 07:43:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   31C    P0             46W /  600W |       3MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [4]:
ram_gb = psutil.virtual_memory().total / 1e9
print('Your runtime has {:.1f} gigabytes of available RAM\n'.format(ram_gb))

if ram_gb < 20:
  print('Not using a high-RAM runtime')
else:
  print('You are using a high-RAM runtime!')

Your runtime has 189.9 gigabytes of available RAM

You are using a high-RAM runtime!


In [5]:
run_start = datetime.datetime.now()
timestamp = run_start.strftime("%Y-%m-%d_%H-%M-%S")
print(f"Current timestamp: {timestamp}")

LOG_DIR = f'/content/drive/MyDrive/ColabThesisData/results/multilingual-e5-large_{timestamp}/'
os.makedirs(LOG_DIR, exist_ok=True)
print(f"Log directory: {LOG_DIR}")

Current timestamp: 2026-04-09_07-43-09
Log directory: /content/drive/MyDrive/ColabThesisData/results/multilingual-e5-large_2026-04-09_07-43-09/


## Load Model

In [6]:
BASE_MODEL = 'TurkuNLP/sbert-cased-finnish-paraphrase'
FINNSENTIMENT_MODEL_PATH = f'/tmp/{BASE_MODEL.split("/")[-1]}_finnsentiment_{timestamp}/'
NUM_LABELS  = 3
LABEL_NAMES = ["negative", "neutral", "positive"]
MAX_SEQ_LENGTH = 512

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

print(f"Tokenizer loaded: {BASE_MODEL}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/617 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/353 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/3.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Tokenizer loaded: TurkuNLP/sbert-cased-finnish-paraphrase


## Define Eval Func

In [7]:
def compute_metrics(eval_pred):
    preds, labels = eval_pred
    preds = np.argmax(preds, axis=1)

    # MAEM: macro-averaged MAE over ordinal classes (Baccianella et al., 2009)
    # Averages per-class MAE to handle class imbalance in ordinal sentiment.
    classes = np.unique(labels)
    maem = float(np.mean([
        np.mean(np.abs(preds[labels == c] - c)) for c in classes
    ]))

    return {
        "accuracy":  accuracy_score(labels, preds),
        "f1":        f1_score(labels, preds, average="weighted", zero_division=0),
        "precision": precision_score(labels, preds, average="weighted", zero_division=0),
        "recall":    recall_score(labels, preds, average="weighted", zero_division=0),
        "maem":      maem,
    }

In [ ]:
def ordinal_emd_loss(logits, labels, class_weights=None):
    """
    Ordinal Earth Mover's Distance (Wasserstein-1) loss for ordered classes.
    Penalizes mistakes in proportion to ordinal distance.
    Labels must be encoded as 0, 1, ..., K-1.
    """
    num_classes = logits.size(-1)

    if labels.dtype != torch.long:
        labels = labels.long()

    probs    = torch.softmax(logits, dim=-1)                          # (B, K)
    cum_pred = torch.cumsum(probs, dim=-1)[..., :-1]                  # (B, K-1)

    cum_true = torch.cumsum(
        torch.nn.functional.one_hot(labels, num_classes=num_classes).float(), dim=-1
    )[..., :-1]                                                        # (B, K-1)

    per_sample = torch.abs(cum_pred - cum_true).sum(dim=-1)           # (B,)

    if class_weights is not None:
        class_weights  = class_weights.to(logits.device)
        sample_weights = class_weights[labels]
        return (per_sample * sample_weights).sum() / sample_weights.sum()

    return per_sample.mean()

## FinnSentiment Pre-training

In [9]:
finnsentiment = pd.read_pickle('/content/drive/MyDrive/ColabThesisData/finnsentiment_cleaned_unambiguous.pkl')

df_0 = finnsentiment[finnsentiment['label'] == 0]
df_1 = finnsentiment[finnsentiment['label'] == 1]
df_2 = finnsentiment[finnsentiment['label'] == 2]
df_1_balanced = df_1.sample(n=min(len(df_0), len(df_2)), random_state=42)
finnsentiment_balanced = pd.concat([df_0, df_1_balanced, df_2]).reset_index(drop=True)

print(f"FinnSentiment balanced: {len(finnsentiment_balanced):,}")
print(finnsentiment_balanced['label'].value_counts().sort_index())

FinnSentiment balanced: 3,818
label
0    1230
1    1230
2    1358
Name: count, dtype: int64


In [10]:
finnsentiment_balanced.sample(5)

,label,text
194,0,Sinä sönkkäät puuta heinää ilman mitään tietoa...
198,0,Ihan hullua porukkaa.............................
2717,2,Rakas!
3766,2,"Onnellinen, etten menettänyt mitään :D"
339,0,Kokkolasta nyt ei löydy kovin järkevää tekemis...


In [11]:
finnsentiment_balanced["label"].value_counts()

,count
label,
2,1358
0,1230
1,1230


In [12]:
fs_model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=NUM_LABELS,
    device_map="auto",
    dtype=torch.bfloat16,
)
print(f"Model loaded from BASE_MODEL for FinnSentiment fine-tuning")

model.safetensors:   0%|          | 0.00/498M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: TurkuNLP/sbert-cased-finnish-paraphrase
Key                     | Status     | 
------------------------+------------+-
embeddings.position_ids | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded from BASE_MODEL for FinnSentiment fine-tuning


In [13]:
# 90/10 train/val split for FinnSentiment
fs_train_df, fs_val_df = train_test_split(
    finnsentiment_balanced, test_size=0.1, random_state=42,
    stratify=finnsentiment_balanced['label'],
)

def make_fs_dataset(df):
    hf = Dataset.from_pandas(df[['text', 'label']].reset_index(drop=True))
    return hf.map(
        lambda batch: tokenizer(batch['text'], truncation=True, max_length=MAX_SEQ_LENGTH),
        batched=True,
    )

fs_train_ds = make_fs_dataset(fs_train_df)
fs_val_ds   = make_fs_dataset(fs_val_df)

Map:   0%|          | 0/3436 [00:00<?, ? examples/s]

Map:   0%|          | 0/382 [00:00<?, ? examples/s]

In [14]:
fs_training_args = TrainingArguments(
    output_dir='/tmp/fs_checkpoints/',
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=75,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="maem",
    greater_is_better=False,
    bf16=torch.cuda.is_available(),
    report_to="none",
    remove_unused_columns=True,
)

fs_trainer = Trainer(
    model=fs_model,
    args=fs_training_args,
    train_dataset=fs_train_ds,
    eval_dataset=fs_val_ds,
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

fs_trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall,Maem
1,No log,0.737911,0.816754,0.816826,0.819323,0.816754,0.218197
2,No log,0.430247,0.900524,0.901097,0.903030,0.900524,0.120776
3,0.718444,0.361648,0.897906,0.898559,0.901071,0.897906,0.125159
4,0.718444,0.345678,0.900524,0.901141,0.903418,0.900524,0.122449
5,0.351070,0.343643,0.900524,0.901141,0.903418,0.900524,0.122449


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=1075, training_loss=0.5200205798481786, metrics={'train_runtime': 22.924, 'train_samples_per_second': 749.434, 'train_steps_per_second': 46.894, 'total_flos': 683535770080128.0, 'train_loss': 0.5200205798481786, 'epoch': 5.0})

In [15]:
fs_trainer.save_model(FINNSENTIMENT_MODEL_PATH)
tokenizer.save_pretrained(FINNSENTIMENT_MODEL_PATH)
print(f"FinnSentiment-tuned model saved to: {FINNSENTIMENT_MODEL_PATH}")

fs_results = fs_trainer.predict(fs_val_ds)
fs_preds = np.argmax(fs_results.predictions, axis=1)
ft_true = fs_val_df["label"].tolist()

print("\n" + "=" * 50)
print("HOLDOUT RESULTS — FINNSENTIMENT-TUNED MODEL")
print("=" * 50)
print(classification_report(ft_true, fs_preds, target_names=LABEL_NAMES))

print("Confusion matrix (rows=true, cols=predicted):")
print(pd.DataFrame(
    confusion_matrix(ft_true, fs_preds),
    index=[f"true_{l}" for l in LABEL_NAMES],
    columns=[f"pred_{l}" for l in LABEL_NAMES],
))

del fs_model, fs_trainer
gc.collect(); torch.cuda.empty_cache()

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

FinnSentiment-tuned model saved to: /tmp/sbert-cased-finnish-paraphrase_finnsentiment_2026-04-09_07-43-09/



HOLDOUT RESULTS — FINNSENTIMENT-TUNED MODEL
              precision    recall  f1-score   support

    negative       0.84      0.91      0.88       123
     neutral       0.89      0.86      0.88       123
    positive       0.97      0.93      0.95       136

    accuracy                           0.90       382
   macro avg       0.90      0.90      0.90       382
weighted avg       0.90      0.90      0.90       382

Confusion matrix (rows=true, cols=predicted):
               pred_negative  pred_neutral  pred_positive
true_negative            112             8              3
true_neutral              16           106              1
true_positive              5             5            126


## Pseudo-label Training

In [ ]:
def make_hf_dataset(df):
    df = df.copy()
    df["text"] = "yritys: " + df["company_name"] + " viesti: " + df["message"]
    hf = Dataset.from_pandas(df[["text", "sentiment"]].rename(columns={"sentiment": "label"}).reset_index(drop=True))
    return hf.map(
        lambda batch: tokenizer(batch["text"], truncation=True, max_length=MAX_SEQ_LENGTH),
        batched=True,
    )

In [17]:
PSEUDO_MODEL_PATH = f'/tmp/{BASE_MODEL.split("/")[-1]}_pseudo_{timestamp}/'
LLM_LABELS_PATH   = '/content/drive/MyDrive/ColabThesisData/llm_labeled.parquet'

llm_labels = pd.read_parquet(LLM_LABELS_PATH)
print(f"LLM pseudo-labels: {len(llm_labels):,}")
print(llm_labels["sentiment"].value_counts().sort_index().rename({0: "negative", 1: "neutral", 2: "positive"}))

pseudo_ds = make_hf_dataset(llm_labels[["message", "sentiment", "company_name"]])

LLM pseudo-labels: 10,000
sentiment
negative    3591
neutral     2782
positive    3627
Name: count, dtype: int64


Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [18]:
pseudo_model = AutoModelForSequenceClassification.from_pretrained(
    FINNSENTIMENT_MODEL_PATH, num_labels=NUM_LABELS, device_map="auto", dtype=torch.bfloat16)

pseudo_args = TrainingArguments(
    output_dir='/tmp/pseudo_ckpts/',
    num_train_epochs=10,
    per_device_train_batch_size=16,
    learning_rate=3e-5,
    weight_decay=0.01,
    warmup_ratio=0.06,
    logging_steps=50,
    save_strategy="no",
    bf16=torch.cuda.is_available(),
    report_to="none",
    remove_unused_columns=True,
)

pseudo_trainer = Trainer(
    model=pseudo_model,
    args=pseudo_args,
    train_dataset=pseudo_ds,
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
)
pseudo_trainer.train()

pseudo_trainer.save_model(PSEUDO_MODEL_PATH)
tokenizer.save_pretrained(PSEUDO_MODEL_PATH)
print(f"\nPseudo-label model saved to: {PSEUDO_MODEL_PATH}")

del pseudo_model, pseudo_trainer
gc.collect(); torch.cuda.empty_cache()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Step,Training Loss
50,1.392233
100,1.241065
150,1.053306
200,0.978192
250,0.934522
300,0.964307
350,0.936927
400,0.929103
450,0.902238
500,0.921735


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Pseudo-label model saved to: /tmp/sbert-cased-finnish-paraphrase_pseudo_2026-04-09_07-43-09/


## Load Human-labeled Financial Data

In [19]:
human_labeled_sentiment = pd.read_parquet('/content/drive/MyDrive/ColabThesisData/labeled.parquet')

# Replace with your dataset loading
ds = human_labeled_sentiment

In [ ]:
ds.sample(5)

,id,author_id,message,date_time,engagement,forum,url,company_name,ticker,message_length,year_month,year,sentiment,labeled_at
109,Sijoitustieto.comment-422,Sijoitustieto.Unknown,"Nordea on kyllä hyvä pankki, mutta minua huole...",2014-07-22 13:41:00.000,N/A,Sijoitustieto,https://www.sijoitustieto.fi/sijoituskeskustel...,Nordea,NDA-FI,255,2014-07,2014,0,2026-03-16T17:18:44.966540
10,Kauppalehti.post-7315597,Kauppalehti.58646,Jos markkinoilla ois sama näkemys kuin tällä p...,2023-12-15 15:53:46.000,N/A,Kauppalehti,https://keskustelu.kauppalehti.fi/threads/ssh-...,SSH Communications Security,SSH1V,133,2023-12,2023,0,2026-03-16T09:39:30.502368
184,Inderes.313187,Inderes.6530,Kyllähän Harvian käyttökatteesta näkee että hi...,2021-07-04 04:44:54.112,38,Inderes,https://forum.inderes.com/t/harvia-foorumi-eli...,Harvia,HARVIA,206,2021-07,2021,1,2026-03-16T17:41:56.446339
77,Kauppalehti.post-7616837,Kauppalehti.57519,Arvuuttelija70 sanoi:\nTämä on tällaista... os...,2025-08-26 12:12:27.000,"Reactions:\nVerot, öölman ja Arvuuttelija70",Kauppalehti,https://keskustelu.kauppalehti.fi/threads/endo...,Endomines Finland,PAMPALO,661,2025-08,2025,1,2026-03-16T14:58:37.056246
538,Sijoitustieto.comment-600,Sijoitustieto.Unknown,Näinhän se on :)....mullakin suurin huoli siin...,2014-08-07 13:51:00.000,N/A,Sijoitustieto,https://www.sijoitustieto.fi/sijoituskeskustel...,UPM-Kymmene,UPM,176,2014-08,2014,1,2026-03-16T20:47:03.335281


In [ ]:
ds["sentiment"].value_counts()

,count
sentiment,
1,253
2,205
0,150


## K-Fold Cross Validation (Final Phase)

### Helper function to save results

In [ ]:
import json as _json

def _to_jsonable(obj):
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    return obj

def _deep_convert(d):
    if isinstance(d, dict):
        return {k: _deep_convert(v) for k, v in d.items()}
    if isinstance(d, list):
        return [_deep_convert(v) for v in d]
    return _to_jsonable(d)

def save_fold_results(label, results, log_dir=None):
    """Persist fold results to Google Drive as JSON + human-readable txt."""
    if log_dir is None:
        log_dir = LOG_DIR
    os.makedirs(log_dir, exist_ok=True)

    safe = label.lower()
    for ch in " /→()—":
        safe = safe.replace(ch, "_")
    while "__" in safe:
        safe = safe.replace("__", "_")
    safe = safe.strip("_")

    # ── JSON (full, machine-readable) ─────────────────────────────────────────
    json_path = os.path.join(log_dir, f"{safe}.json")
    with open(json_path, "w") as f:
        _json.dump({"label": label, "folds": _deep_convert(results)}, f, indent=2)

    # ── TXT (human-readable summary) ──────────────────────────────────────────
    txt_path = os.path.join(log_dir, f"{safe}.txt")
    accs  = [r["accuracy"]    for r in results]
    f1w   = [r["f1_weighted"] for r in results]
    f1m   = [r["f1_macro"]    for r in results]
    maems = [r.get("maem", float("nan")) for r in results]

    lines = [
        "=" * 62,
        f"  {label}",
        "=" * 62,
        "",
        "Per-fold results:",
    ]
    for i, r in enumerate(results):
        lines.append(
            f"  Fold {i+1}: acc={r['accuracy']:.4f}  f1_w={r['f1_weighted']:.4f}"
            f"  f1_macro={r['f1_macro']:.4f}  maem={r.get('maem', float('nan')):.4f}"
        )
    lines += [
        "",
        "Mean ± Std:",
        f"  Accuracy    : {np.mean(accs):.4f} ± {np.std(accs):.4f}",
        f"  F1 weighted : {np.mean(f1w):.4f} ± {np.std(f1w):.4f}",
        f"  F1 macro    : {np.mean(f1m):.4f} ± {np.std(f1m):.4f}",
        f"  MAEM        : {np.mean(maems):.4f} ± {np.std(maems):.4f}",
        "",
        "Mean per-class metrics:",
    ]
    for cls in LABEL_NAMES:
        prec = np.mean([r["report"][cls]["precision"] for r in results])
        rec  = np.mean([r["report"][cls]["recall"]    for r in results])
        f1   = np.mean([r["report"][cls]["f1-score"]  for r in results])
        lines.append(f"  {cls:>10}: precision={prec:.4f}  recall={rec:.4f}  f1={f1:.4f}")

    agg_cm = sum(np.array(r["confusion"]) for r in results)
    lines += ["", "Aggregated confusion matrix:"]
    lines.append("               " + "  ".join(f"pred_{l}" for l in LABEL_NAMES))
    for row_l, row in zip(LABEL_NAMES, agg_cm):
        lines.append(f"  true_{row_l:>8}: " + "  ".join(f"{int(v):6d}" for v in row))

    with open(txt_path, "w") as f:
        f.write("\n".join(lines) + "\n")

    print(f"  [logged] {json_path}")
    print(f"  [logged] {txt_path}")

In [ ]:
N_FOLDS = 5

cv_df = ds[["id", "message", "sentiment", "company_name"]].reset_index(drop=True)

kf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

fold_results = []

In [24]:
for fold, (train_idx, val_idx) in enumerate(kf.split(cv_df, cv_df["sentiment"])):
    print(f"\n{'='*55}")
    print(f"  FOLD {fold+1}/{N_FOLDS}")
    print(f"{'='*55}")

    fold_train_df = cv_df.iloc[train_idx][["message", "sentiment", "company_name"]]
    fold_val_df   = cv_df.iloc[val_idx][["message", "sentiment", "company_name"]]
    fold_true     = fold_val_df["sentiment"].tolist()

    print(f"Train: {len(fold_train_df)}  |  Val: {len(fold_val_df)}")

    fold_train_ds = make_hf_dataset(fold_train_df)
    fold_val_ds   = make_hf_dataset(fold_val_df)

    fold_model = AutoModelForSequenceClassification.from_pretrained(
        PSEUDO_MODEL_PATH, num_labels=NUM_LABELS, device_map="auto", dtype=torch.bfloat16)

    fold_cw = compute_class_weight('balanced', classes=np.array([0, 1, 2]), y=fold_train_df['sentiment'].values)
    fold_cw_tensor = torch.tensor(fold_cw, dtype=torch.float).to(fold_model.device)

    class FoldWeightedTrainer(Trainer):
        def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
            labels = inputs.pop("labels")
            outputs = model(**inputs)
            loss = ordinal_emd_loss(outputs.logits, labels, class_weights=fold_cw_tensor)
            return (loss, outputs) if return_outputs else loss

    fold_ft_args = TrainingArguments(
        output_dir=f'/tmp/fold_{fold+1}_ft_ckpts/',
        num_train_epochs=10,
        per_device_train_batch_size=16, per_device_eval_batch_size=32,
        learning_rate=3e-5, weight_decay=0.01, warmup_ratio=0.1, logging_steps=5,
        eval_strategy="epoch", save_strategy="epoch", save_total_limit=1,
        load_best_model_at_end=True, metric_for_best_model="maem", greater_is_better=False,
        bf16=torch.cuda.is_available(), report_to="none", remove_unused_columns=True,
    )
    fold_trainer = FoldWeightedTrainer(
        model=fold_model, args=fold_ft_args,
        train_dataset=fold_train_ds, eval_dataset=fold_val_ds, processing_class=tokenizer,
        data_collator=DataCollatorWithPadding(tokenizer=tokenizer), compute_metrics=compute_metrics,
    )
    fold_trainer.train()

    fold_preds = np.argmax(fold_trainer.predict(fold_val_ds).predictions, axis=1)
    _ft   = np.array(fold_true)
    _maem = float(np.mean([np.mean(np.abs(fold_preds[_ft == c] - c)) for c in np.unique(_ft)]))
    fold_results.append({
        "accuracy":    accuracy_score(fold_true, fold_preds),
        "f1_weighted": f1_score(fold_true, fold_preds, average="weighted", zero_division=0),
        "f1_macro":    f1_score(fold_true, fold_preds, average="macro",    zero_division=0),
        "maem":        _maem,
        "report":      classification_report(fold_true, fold_preds, target_names=LABEL_NAMES, output_dict=True, zero_division=0),
        "confusion":   confusion_matrix(fold_true, fold_preds),
    })
    print(f"  acc={fold_results[-1]['accuracy']:.3f}  f1_weighted={fold_results[-1]['f1_weighted']:.3f}  f1_macro={fold_results[-1]['f1_macro']:.3f}  maem={fold_results[-1]['maem']:.3f}")

    del fold_model, fold_trainer
    gc.collect(); torch.cuda.empty_cache()


  FOLD 1/5
Train: 486  |  Val: 122


Map:   0%|          | 0/486 [00:00<?, ? examples/s]

Map:   0%|          | 0/122 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

  acc=0.612  f1_weighted=0.606  f1_macro=0.608  maem=0.475


In [25]:
accs  = [r["accuracy"]    for r in fold_results]
f1w   = [r["f1_weighted"] for r in fold_results]
f1m   = [r["f1_macro"]    for r in fold_results]
maems = [r.get("maem", float("nan")) for r in fold_results]

print("── Per-fold Results ──")
for i, r in enumerate(fold_results):
    print(f"  Fold {i+1}: acc={r['accuracy']:.3f}  f1_weighted={r['f1_weighted']:.3f}  f1_macro={r['f1_macro']:.3f}  maem={r.get('maem', float('nan')):.3f}")

print(f"\n── Mean ± Std ──")
print(f"  Accuracy    : {np.mean(accs):.3f} ± {np.std(accs):.3f}")
print(f"  F1 weighted : {np.mean(f1w):.3f} ± {np.std(f1w):.3f}")
print(f"  F1 macro    : {np.mean(f1m):.3f} ± {np.std(f1m):.3f}")
print(f"  MAEM        : {np.mean(maems):.3f} ± {np.std(maems):.3f}")

print(f"\n── Mean Per-class Metrics (across folds) ──")
for cls in LABEL_NAMES:
    prec = np.mean([r["report"][cls]["precision"] for r in fold_results])
    rec  = np.mean([r["report"][cls]["recall"]    for r in fold_results])
    f1   = np.mean([r["report"][cls]["f1-score"]  for r in fold_results])
    print(f"  {cls:>10}: precision={prec:.3f}  recall={rec:.3f}  f1={f1:.3f}")

agg_cm = sum(r["confusion"] for r in fold_results)
print(f"\n── Aggregated Confusion Matrix (all folds) ──")
print(pd.DataFrame(
    agg_cm,
    index=[f"true_{l}"  for l in LABEL_NAMES],
    columns=[f"pred_{l}" for l in LABEL_NAMES],
))

save_fold_results("Full pipeline (DAPT → FinnSentiment → Pseudo)", fold_results)

── Per-fold Results ──
  Fold 1: acc=0.631  f1_weighted=0.631  f1_macro=0.627  maem=0.468
  Fold 2: acc=0.598  f1_weighted=0.598  f1_macro=0.592  maem=0.487
  Fold 3: acc=0.590  f1_weighted=0.591  f1_macro=0.586  maem=0.530
  Fold 4: acc=0.579  f1_weighted=0.574  f1_macro=0.560  maem=0.561
  Fold 5: acc=0.612  f1_weighted=0.606  f1_macro=0.608  maem=0.475

── Mean ± Std ──
  Accuracy    : 0.602 ± 0.018
  F1 weighted : 0.600 ± 0.019
  F1 macro    : 0.595 ± 0.023
  MAEM        : 0.504 ± 0.036

── Mean Per-class Metrics (across folds) ──
    negative: precision=0.559  recall=0.573  f1=0.562
     neutral: precision=0.626  recall=0.625  f1=0.625
    positive: precision=0.605  recall=0.595  f1=0.598

── Aggregated Confusion Matrix (all folds) ──
               pred_negative  pred_neutral  pred_positive
true_negative             86            35             29
true_neutral              44           158             51
true_positive             23            60            122
  [logged] /conten

## Final Model (All Data)

In [26]:
FINAL_MODEL_PATH = f'/content/drive/MyDrive/ColabThesisData/models/{BASE_MODEL.split("/")[-1]}_final_{timestamp}/'

all_human_df   = ds[["message", "sentiment", "company_name"]].copy()
final_train_ds = make_hf_dataset(all_human_df)

print(f"Final fine-tuning on {len(all_human_df):,} human-labeled samples")
print(all_human_df["sentiment"].value_counts().sort_index().rename({0: "negative", 1: "neutral", 2: "positive"}))

final_model = AutoModelForSequenceClassification.from_pretrained(
    PSEUDO_MODEL_PATH, num_labels=NUM_LABELS, device_map="auto", dtype=torch.bfloat16)

final_cw = compute_class_weight('balanced', classes=np.array([0, 1, 2]), y=all_human_df['sentiment'].values)
final_cw_tensor = torch.tensor(final_cw, dtype=torch.float).to(final_model.device)

class FinalWeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = ordinal_emd_loss(outputs.logits, labels, class_weights=final_cw_tensor)
        return (loss, outputs) if return_outputs else loss

final_args = TrainingArguments(
    output_dir='/tmp/final_ckpts/',
    num_train_epochs=10,
    per_device_train_batch_size=16,
    learning_rate=3e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    logging_steps=5,
    save_strategy="no",
    bf16=torch.cuda.is_available(),
    report_to="none",
    remove_unused_columns=True,
)

final_trainer = FinalWeightedTrainer(
    model=final_model,
    args=final_args,
    train_dataset=final_train_ds,
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
)
final_trainer.train()

final_trainer.save_model(FINAL_MODEL_PATH)
tokenizer.save_pretrained(FINAL_MODEL_PATH)
print(f"Final model saved to: {FINAL_MODEL_PATH}")

Map:   0%|          | 0/608 [00:00<?, ? examples/s]

Final fine-tuning on 608 human-labeled samples
sentiment
negative    150
neutral     253
positive    205
Name: count, dtype: int64


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Step,Training Loss
5,0.665160
10,0.596173
15,0.673673
20,0.610812
25,0.652066
30,0.549353
35,0.601008
40,0.581965
45,0.612415
50,0.644165


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Final model saved to: /content/drive/MyDrive/ColabThesisData/models/sbert-cased-finnish-paraphrase_final_2026-04-09_07-43-09/


## Ablation Studies

Each ablation removes one or more phases from the full pipeline (FinnSentiment → Pseudo → K-Fold) and runs K-Fold CV with the remaining phases.

| Ablation | Phase 1 | K-Fold from |
|---|---|---|
| 1 — Pseudo only | Pseudo (from BASE_MODEL) | Pseudo ckpt |
| 2 — No Pseudo | FinnSentiment (reused) | FinnSentiment ckpt |

In [ ]:
def print_ablation_summary(label, results):
    accs  = [r["accuracy"]    for r in results]
    f1w   = [r["f1_weighted"] for r in results]
    f1m   = [r["f1_macro"]    for r in results]
    maems = [r.get("maem", float("nan")) for r in results]
    print(f"\n{'='*55}")
    print(f"  {label}")
    print(f"{'='*55}")
    for i, r in enumerate(results):
        print(f"  Fold {i+1}: acc={r['accuracy']:.3f}  f1_w={r['f1_weighted']:.3f}  f1_macro={r['f1_macro']:.3f}  maem={r.get('maem', float('nan')):.3f}")
    print(f"\n  Mean ± Std:")
    print(f"    Accuracy    : {np.mean(accs):.3f} ± {np.std(accs):.3f}")
    print(f"    F1 weighted : {np.mean(f1w):.3f} ± {np.std(f1w):.3f}")
    print(f"    F1 macro    : {np.mean(f1m):.3f} ± {np.std(f1m):.3f}")
    print(f"    MAEM        : {np.mean(maems):.3f} ± {np.std(maems):.3f}")
    print(f"\n  Mean Per-class Metrics:")
    for cls in LABEL_NAMES:
        prec = np.mean([r["report"][cls]["precision"] for r in results])
        rec  = np.mean([r["report"][cls]["recall"]    for r in results])
        f1   = np.mean([r["report"][cls]["f1-score"]  for r in results])
        print(f"    {cls:>10}: precision={prec:.3f}  recall={rec:.3f}  f1={f1:.3f}")
    agg_cm = sum(r["confusion"] for r in results)
    print(f"\n  Aggregated Confusion Matrix:")
    print(pd.DataFrame(agg_cm,
        index=[f"true_{l}" for l in LABEL_NAMES],
        columns=[f"pred_{l}" for l in LABEL_NAMES]))
    save_fold_results(label, results)

### Ablation 1 — Pseudo Only: Base → Pseudo → K-Fold

Skips FinnSentiment entirely. Loads `BASE_MODEL` directly as a classifier and trains only on pseudo-labeled data, isolating the contribution of pseudo-labeling.

In [28]:
ABL2_PSEUDO_PATH = f'/tmp/{BASE_MODEL.split("/")[-1]}_abl1_nofs_pseudo_{timestamp}/'

# Load BASE_MODEL directly as classifier — skip FinnSentiment
abl2_pseudo_model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL, num_labels=NUM_LABELS, device_map="auto", dtype=torch.bfloat16)

abl2_pseudo_args = TrainingArguments(
    output_dir='/tmp/abl1_pseudo_ckpts/',
    num_train_epochs=10,
    per_device_train_batch_size=16,
    learning_rate=3e-5, weight_decay=0.01, warmup_ratio=0.06,
    logging_steps=50, save_strategy="no",
    bf16=torch.cuda.is_available(), report_to="none", remove_unused_columns=True,
)
abl2_pseudo_trainer = Trainer(
    model=abl2_pseudo_model, args=abl2_pseudo_args,
    train_dataset=pseudo_ds,
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
)
abl2_pseudo_trainer.train()
abl2_pseudo_trainer.save_model(ABL2_PSEUDO_PATH)
tokenizer.save_pretrained(ABL2_PSEUDO_PATH)
print(f"Ablation 1 — Pseudo model saved: {ABL2_PSEUDO_PATH}")

del abl2_pseudo_model, abl2_pseudo_trainer
gc.collect(); torch.cuda.empty_cache()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: TurkuNLP/sbert-cased-finnish-paraphrase
Key                     | Status     | 
------------------------+------------+-
embeddings.position_ids | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Step,Training Loss
50,1.165874
100,1.128555
150,1.115581
200,1.095059
250,1.067729
300,1.066230
350,1.037310
400,1.016091
450,0.960919
500,0.953341


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Ablation 1 — Pseudo model saved: /tmp/sbert-cased-finnish-paraphrase_abl1_nofs_pseudo_2026-04-09_07-43-09/


In [29]:
abl2_fold_results = []

for fold, (train_idx, val_idx) in enumerate(kf.split(cv_df, cv_df["sentiment"])):
    print(f"\n{'='*55}\n  FOLD {fold+1}/{N_FOLDS}\n{'='*55}")

    fold_train_df = cv_df.iloc[train_idx][["message", "sentiment", "company_name"]]
    fold_val_df   = cv_df.iloc[val_idx][["message", "sentiment", "company_name"]]
    fold_true     = fold_val_df["sentiment"].tolist()
    print(f"Train: {len(fold_train_df)}  |  Val: {len(fold_val_df)}")

    fold_train_ds = make_hf_dataset(fold_train_df)
    fold_val_ds   = make_hf_dataset(fold_val_df)

    fold_model = AutoModelForSequenceClassification.from_pretrained(
        ABL2_PSEUDO_PATH, num_labels=NUM_LABELS, device_map="auto", dtype=torch.bfloat16)

    fold_cw = compute_class_weight('balanced', classes=np.array([0,1,2]), y=fold_train_df['sentiment'].values)
    fold_cw_tensor = torch.tensor(fold_cw, dtype=torch.float).to(fold_model.device)

    class Abl2FoldTrainer(Trainer):
        def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
            labels = inputs.pop("labels")
            outputs = model(**inputs)
            loss = ordinal_emd_loss(outputs.logits, labels, class_weights=fold_cw_tensor)
            return (loss, outputs) if return_outputs else loss

    fold_args = TrainingArguments(
        output_dir=f'/tmp/abl2_fold_{fold+1}_ckpts/',
        num_train_epochs=10,
        per_device_train_batch_size=16, per_device_eval_batch_size=32,
        learning_rate=3e-5, weight_decay=0.01, warmup_ratio=0.1, logging_steps=5,
        eval_strategy="epoch", save_strategy="epoch", save_total_limit=1,
        load_best_model_at_end=True, metric_for_best_model="maem", greater_is_better=False,
        bf16=torch.cuda.is_available(), report_to="none", remove_unused_columns=True,
    )
    fold_trainer = Abl2FoldTrainer(
        model=fold_model, args=fold_args,
        train_dataset=fold_train_ds, eval_dataset=fold_val_ds,
        processing_class=tokenizer,
        data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
        compute_metrics=compute_metrics,
    )
    fold_trainer.train()

    fold_preds = np.argmax(fold_trainer.predict(fold_val_ds).predictions, axis=1)
    _ft   = np.array(fold_true)
    _maem = float(np.mean([np.mean(np.abs(fold_preds[_ft == c] - c)) for c in np.unique(_ft)]))
    abl2_fold_results.append({
        "accuracy":    accuracy_score(fold_true, fold_preds),
        "f1_weighted": f1_score(fold_true, fold_preds, average="weighted", zero_division=0),
        "f1_macro":    f1_score(fold_true, fold_preds, average="macro",    zero_division=0),
        "maem":        _maem,
        "report":      classification_report(fold_true, fold_preds, target_names=LABEL_NAMES, output_dict=True, zero_division=0),
        "confusion":   confusion_matrix(fold_true, fold_preds),
    })
    print(f"  acc={abl2_fold_results[-1]['accuracy']:.3f}  f1_weighted={abl2_fold_results[-1]['f1_weighted']:.3f}  f1_macro={abl2_fold_results[-1]['f1_macro']:.3f}  maem={abl2_fold_results[-1]['maem']:.3f}")

    del fold_model, fold_trainer
    gc.collect(); torch.cuda.empty_cache()

print_ablation_summary("ABLATION 1 — Pseudo Only (Base → Pseudo → K-Fold)", abl2_fold_results)


  FOLD 1/5
Train: 486  |  Val: 122


Map:   0%|          | 0/486 [00:00<?, ? examples/s]

Map:   0%|          | 0/122 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

  acc=0.595  f1_weighted=0.592  f1_macro=0.592  maem=0.505

  ABLATION 1 — Pseudo Only (Base → Pseudo → K-Fold)
  Fold 1: acc=0.664  f1_w=0.656  f1_macro=0.639  maem=0.463
  Fold 2: acc=0.639  f1_w=0.637  f1_macro=0.626  maem=0.455
  Fold 3: acc=0.574  f1_w=0.573  f1_macro=0.566  maem=0.547
  Fold 4: acc=0.521  f1_w=0.515  f1_macro=0.499  maem=0.611
  Fold 5: acc=0.595  f1_w=0.592  f1_macro=0.592  maem=0.505

  Mean ± Std:
    Accuracy    : 0.599 ± 0.050
    F1 weighted : 0.595 ± 0.050
    F1 macro    : 0.584 ± 0.050
    MAEM        : 0.517 ± 0.058

  Mean Per-class Metrics:
      negative: precision=0.562  recall=0.487  f1=0.519
       neutral: precision=0.599  recall=0.687  f1=0.640
      positive: precision=0.621  recall=0.571  f1=0.594

  Aggregated Confusion Matrix:
               pred_negative  pred_neutral  pred_positive
true_negative             73            46             31
true_neutral              38           174             41
true_positive             18            70  

### Ablation 2 — No Pseudo: FinnSentiment → K-Fold

Reuses `FINNSENTIMENT_MODEL_PATH` from the main pipeline — no re-training needed.

In [30]:
abl3_fold_results = []

for fold, (train_idx, val_idx) in enumerate(kf.split(cv_df, cv_df["sentiment"])):
    print(f"\n{'='*55}\n  FOLD {fold+1}/{N_FOLDS}\n{'='*55}")

    fold_train_df = cv_df.iloc[train_idx][["message", "sentiment", "company_name"]]
    fold_val_df   = cv_df.iloc[val_idx][["message", "sentiment", "company_name"]]
    fold_true     = fold_val_df["sentiment"].tolist()
    print(f"Train: {len(fold_train_df)}  |  Val: {len(fold_val_df)}")

    fold_train_ds = make_hf_dataset(fold_train_df)
    fold_val_ds   = make_hf_dataset(fold_val_df)

    fold_model = AutoModelForSequenceClassification.from_pretrained(
        FINNSENTIMENT_MODEL_PATH, num_labels=NUM_LABELS, device_map="auto", dtype=torch.bfloat16)

    fold_cw = compute_class_weight('balanced', classes=np.array([0,1,2]), y=fold_train_df['sentiment'].values)
    fold_cw_tensor = torch.tensor(fold_cw, dtype=torch.float).to(fold_model.device)

    class Abl3FoldTrainer(Trainer):
        def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
            labels = inputs.pop("labels")
            outputs = model(**inputs)
            loss = ordinal_emd_loss(outputs.logits, labels, class_weights=fold_cw_tensor)
            return (loss, outputs) if return_outputs else loss

    fold_args = TrainingArguments(
        output_dir=f'/tmp/abl3_fold_{fold+1}_ckpts/',
        num_train_epochs=10,
        per_device_train_batch_size=16, per_device_eval_batch_size=32,
        learning_rate=3e-5, weight_decay=0.01, warmup_ratio=0.1, logging_steps=5,
        eval_strategy="epoch", save_strategy="epoch", save_total_limit=1,
        load_best_model_at_end=True, metric_for_best_model="maem", greater_is_better=False,
        bf16=torch.cuda.is_available(), report_to="none", remove_unused_columns=True,
    )
    fold_trainer = Abl3FoldTrainer(
        model=fold_model, args=fold_args,
        train_dataset=fold_train_ds, eval_dataset=fold_val_ds,
        processing_class=tokenizer,
        data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
        compute_metrics=compute_metrics,
    )
    fold_trainer.train()

    fold_preds = np.argmax(fold_trainer.predict(fold_val_ds).predictions, axis=1)
    _ft   = np.array(fold_true)
    _maem = float(np.mean([np.mean(np.abs(fold_preds[_ft == c] - c)) for c in np.unique(_ft)]))
    abl3_fold_results.append({
        "accuracy":    accuracy_score(fold_true, fold_preds),
        "f1_weighted": f1_score(fold_true, fold_preds, average="weighted", zero_division=0),
        "f1_macro":    f1_score(fold_true, fold_preds, average="macro",    zero_division=0),
        "maem":        _maem,
        "report":      classification_report(fold_true, fold_preds, target_names=LABEL_NAMES, output_dict=True, zero_division=0),
        "confusion":   confusion_matrix(fold_true, fold_preds),
    })
    print(f"  acc={abl3_fold_results[-1]['accuracy']:.3f}  f1_weighted={abl3_fold_results[-1]['f1_weighted']:.3f}  f1_macro={abl3_fold_results[-1]['f1_macro']:.3f}  maem={abl3_fold_results[-1]['maem']:.3f}")

    del fold_model, fold_trainer
    gc.collect(); torch.cuda.empty_cache()

print_ablation_summary("ABLATION 2 — No Pseudo (FinnSentiment → K-Fold)", abl3_fold_results)


  FOLD 1/5
Train: 486  |  Val: 122


Map:   0%|          | 0/486 [00:00<?, ? examples/s]

Map:   0%|          | 0/122 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

  acc=0.413  f1_weighted=0.242  f1_macro=0.195  maem=0.667

  ABLATION 2 — No Pseudo (FinnSentiment → K-Fold)
  Fold 1: acc=0.467  f1_w=0.368  f1_macro=0.317  maem=0.634
  Fold 2: acc=0.459  f1_w=0.357  f1_macro=0.305  maem=0.642
  Fold 3: acc=0.410  f1_w=0.305  f1_macro=0.257  maem=0.686
  Fold 4: acc=0.438  f1_w=0.314  f1_macro=0.266  maem=0.662
  Fold 5: acc=0.413  f1_w=0.242  f1_macro=0.195  maem=0.667

  Mean ± Std:
    Accuracy    : 0.437 ± 0.023
    F1 weighted : 0.317 ± 0.045
    F1 macro    : 0.268 ± 0.043
    MAEM        : 0.658 ± 0.019

  Mean Per-class Metrics:
      negative: precision=0.000  recall=0.000  f1=0.000
       neutral: precision=0.427  recall=0.933  f1=0.586
      positive: precision=0.450  recall=0.146  f1=0.218

  Aggregated Confusion Matrix:
               pred_negative  pred_neutral  pred_positive
true_negative              0           142              8
true_neutral               1           236             16
true_positive              0           175    

### Ablation Comparison

In [ ]:
def mean_f1m(results): return np.mean([r["f1_macro"] for r in results])
def mean_f1w(results): return np.mean([r["f1_weighted"] for r in results])
def mean_acc(results): return np.mean([r["accuracy"] for r in results])
def std_f1m(results):  return np.std([r["f1_macro"] for r in results])
def mean_maem(results): return np.mean([r.get("maem", float("nan")) for r in results])
def std_maem(results):  return np.std([r.get("maem", float("nan")) for r in results])

full_fold_results = fold_results

rows = [
    ("Full pipeline (FS → Pseudo)",      full_fold_results),
    ("Ablation 1 — Pseudo only",          abl2_fold_results),
    ("Ablation 2 — No Pseudo (FS only)", abl3_fold_results),
]

print(f"\n{'='*85}")
print(f"  {'Pipeline':<42} {'Acc':>6}  {'F1-w':>6}  {'F1-macro':>8}  {'±std':>6}  {'MAEM':>6}  {'±std':>6}")
print(f"{'='*85}")
for name, res in rows:
    print(f"  {name:<42} {mean_acc(res):.3f}   {mean_f1w(res):.3f}   {mean_f1m(res):.3f}     ±{std_f1m(res):.3f}  {mean_maem(res):.3f}  ±{std_maem(res):.3f}")
print(f"{'='*85}")

# ── Save comparison table as CSV ─────────────────────────────────────────
import csv as _csv
csv_path = os.path.join(LOG_DIR, "ablation_comparison.csv")
with open(csv_path, "w", newline="") as _f:
    writer = _csv.writer(_f)
    writer.writerow(["pipeline", "acc_mean", "acc_std", "f1w_mean", "f1w_std",
                     "f1m_mean", "f1m_std", "maem_mean", "maem_std"])
    for name, res in rows:
        writer.writerow([
            name,
            f"{mean_acc(res):.4f}", f"{np.std([r['accuracy'] for r in res]):.4f}",
            f"{mean_f1w(res):.4f}", f"{np.std([r['f1_weighted'] for r in res]):.4f}",
            f"{mean_f1m(res):.4f}", f"{std_f1m(res):.4f}",
            f"{mean_maem(res):.4f}", f"{std_maem(res):.4f}",
        ])
print(f"\n[logged] {csv_path}")


  Pipeline                                      Acc    F1-w  F1-macro    ±std    MAEM    ±std
  Full pipeline (FS → Pseudo)                0.602   0.600   0.595     ±0.023  0.504  ±0.036
  Ablation 1 — Pseudo only                   0.599   0.595   0.584     ±0.050  0.517  ±0.058
  Ablation 2 — No Pseudo (FS only)           0.437   0.317   0.268     ±0.043  0.658  ±0.019

[logged] /content/drive/MyDrive/ColabThesisData/results/multilingual-e5-large_2026-04-09_07-43-09/ablation_comparison.csv


In [ ]:
final_elapsed = datetime.datetime.now() - run_start
total_seconds = int(final_elapsed.total_seconds())
runtime_str = f"{total_seconds // 3600}h {(total_seconds % 3600) // 60}m {total_seconds % 60}s"
print(f"Pipeline runtime: {runtime_str}")

runtime_log_path = os.path.join(LOG_DIR, "runtime.txt")
with open(runtime_log_path, "w") as _f:
    _f.write(f"Run started : {run_start.strftime('%Y-%m-%d %H:%M:%S')}\n")
    _f.write(f"Run finished: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    _f.write(f"Total runtime: {runtime_str}\n")
print(f"[logged] {runtime_log_path}")

Pipeline runtime: 0h 14m 6s
[logged] /content/drive/MyDrive/ColabThesisData/results/multilingual-e5-large_2026-04-09_07-43-09/runtime.txt
